# 03. Observation Schema Freeze

本 notebook 从 02 的执行 artifacts 冻结候选 observation schema，而不改变任何 feature definition、filter、threshold 或 extractor。当前可执行 Baseline 是 **14D**，不是请求中假设的 18D；23D 是 candidate feature library，不自动等于 23 个 fitting targets。

`Constructive prefix identity` 只说明 `X_augmented=[X_baseline, X_new]` 的构造正确，不是独立 extractor 回归验证。进入 pilot 前仍需 simulator-observable parity、fixed aggregation/scaling/validity policy，以及 inference 与 held-out evidence 的预先分离。

In [1]:
from __future__ import annotations
import json
import os
from pathlib import Path
import sys
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / ".git").exists() and (PROJECT_ROOT / "S4_sbi").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate repository root")
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "S4_sbi" / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from sleep_sbi import overnight_ablation as oa

print("project root:", PROJECT_ROOT)
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV"))
print("sys.executable:", sys.executable)
print("sys.prefix:", sys.prefix)
print("Python:", sys.version)
print("workflow version:", oa.SCHEMA_VERSION)
assert os.environ.get("CONDA_DEFAULT_ENV") == "neurolib"
assert "neurolib" in sys.executable.lower()
assert "neurolib" in sys.prefix.lower()

project root: D:\Year3_Mao_Projects\sleep_loop
CONDA_DEFAULT_ENV: neurolib
sys.executable: C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib\python.exe
sys.prefix: C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib
Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:36:49) [MSC v.1944 64 bit (AMD64)]
workflow version: overnight-observation-ablation-v0.1


## Re-read notebook-02 evidence

先重新读取 schema、support 和 vectors。保持 `142 retained + 78 rejected = 220 N3` 的 QC 恒等式；low event support 或 undefined 不能填零。

In [2]:
comparison = oa.load_schema_comparison()
print("Actual executable Baseline dimension:", len(comparison["baseline_names"]))
print("Candidate Augmented dimension:", len(comparison["augmented_names"]))
print("Constructive prefix identity:", comparison["augmented_names"][:14] == comparison["baseline_names"])
print("Epoch accounting: 142 retained + 78 rejected = 220 N3")
assert len(comparison["baseline_names"]) == oa.ACTUAL_BASELINE_DIMENSION == 14
assert len(comparison["augmented_names"]) == oa.ACTUAL_AUGMENTED_DIMENSION == 23
assert oa.RETAINED_N3_EPOCHS + oa.REJECTED_N3_EPOCHS == oa.ALL_N3_EPOCHS
display(comparison["support"])

Actual executable Baseline dimension: 14
Candidate Augmented dimension: 23
Constructive prefix identity: True
Epoch accounting: 142 retained + 78 rejected = 220 N3


,feature_name,valid_epoch_count,valid_epoch_percentage,missing_epoch_count,missing_percentage,valid_event_count,median_event_support,minimum_support_requirement,most_common_invalid_reason,zero_meaning,status
0,fooof_aperiodic_exponent,142,100.000000,0,0.000000,NaN,NaN,detector/estimator valid for epoch,none/not applicable,0 is a numeric value only; never a missing-val...,Baseline preserved
1,so_peak_frequency_hz,142,100.000000,0,0.000000,NaN,NaN,detector/estimator valid for epoch,none/not applicable,0 is a numeric value only; never a missing-val...,Baseline preserved
2,relative_so_power,142,100.000000,0,0.000000,NaN,NaN,detector/estimator valid for epoch,none/not applicable,0 is a numeric value only; never a missing-val...,Baseline preserved
3,so_q,142,100.000000,0,0.000000,NaN,NaN,detector/estimator valid for epoch,none/not applicable,0 is a numeric value only; never a missing-val...,Baseline preserved
4,so_event_rate_per_min,142,100.000000,0,0.000000,931.0,5.0,detector/estimator valid for epoch,none/not applicable,0 = no detected event in a detector-valid epoc...,Baseline preserved
5,ibi_cv,124,87.323944,18,12.676056,779.0,5.0,>=2 within-epoch intervals (>=3 SO events),fewer_than_three_so_events,0 is a numeric value only; never a missing-val...,Baseline preserved
6,pac_up_down_ratio,142,100.000000,0,0.000000,NaN,NaN,detector/estimator valid for epoch,none/not applicable,0 is a numeric value only; never a missing-val...,Baseline preserved
7,spindle_density_per_min,142,100.000000,0,0.000000,203.0,1.0,detector/estimator valid for epoch,none/not applicable,0 = no detected spindle in a detector-valid epoch,Baseline preserved
8,spindle_mean_duration_s,124,87.323944,18,12.676056,203.0,2.0,>=1 detected spindle,no_spindle_events_mean_duration_undefined,0 is a numeric value only; never a missing-val...,Baseline preserved
9,pac_mi,142,100.000000,0,0.000000,NaN,NaN,detector/estimator valid for epoch,none/not applicable,0 is a numeric value only; never a missing-val...,Baseline preserved


## Freeze decision

每个 field 的 support、missingness meaning、event support、derived/redundancy relationship、role 和 simulation extractability 都进入正式表。高相关性不会自动删除 field；deterministic/near-deterministic field 与 10/142 support 的 event phase 不能被静默提升为 input。

In [3]:
freeze_result = oa.run_schema_freeze()
decisions = freeze_result["decisions"]
evaluations = freeze_result["evaluations"]
display(oa.concise_status_frame(decisions, ["feature", "group", "real_support", "role", "freeze_status", "redundancy", "rationale"]))
display(oa.concise_status_frame(evaluations, ["schema_id", "dimension", "status", "simulation_extractable", "aggregation_frozen", "scaling_frozen", "held_out_leakage_resolved", "rationale"]))
print("Freeze artifact directory:", freeze_result["output_dir"])

,feature,group,real_support,role,freeze_status,redundancy,rationale
0,fooof_aperiodic_exponent,Baseline spectral/SO,142/142,inference candidate,Conditional parity-only,none identified in current audit,Stable record-level spectral observable; still...
1,so_peak_frequency_hz,Baseline spectral/SO,142/142,inference candidate,Conditional parity-only,related to SO timing but not deterministic,Record-level finite; requires same PSD semanti...
2,relative_so_power,Baseline spectral/SO,142/142,inference candidate,Conditional parity-only,correlated with so_q in current data,Current Hann PSD definition is explicit; simul...
3,so_q,Baseline spectral/SO,142/142,inference candidate,Conditional parity-only,related to relative_so_power,Do not automatically remove correlation; parit...
4,so_event_rate_per_min,SO rhythm,142/142,inference candidate,Conditional parity-only,near-derived with median IBI,"Detector-valid zero has a defined meaning, but..."
5,ibi_cv,SO rhythm,124/142,inference candidate,Conditional parity-only,"related to SO rate/IBI, not deterministic",18/142 epochs lack enough intervals; a simulat...
6,pac_up_down_ratio,SOspindle coordination / PAC,142/142,mechanism diagnostic,Exclude from pilot input,PAC-derived ratio,Observation-level ratio is not a direct intern...
7,spindle_density_per_min,Spindle,142/142,held-out PPC candidate,Hold out pending protocol,approximately linked with occupancy and duration,Observable detector is provisional; retain for...
8,spindle_mean_duration_s,Spindle,124/142,held-out PPC candidate,Hold out pending protocol,approximately linked with occupancy and density,Undefined for no-event epochs; must not be zer...
9,pac_mi,SOspindle coordination / PAC,142/142,held-out PPC candidate,Hold out,PAC family,Preferred phase is unstable when MI is weak; k...


,schema_id,dimension,status,simulation_extractable,aggregation_frozen,scaling_frozen,held_out_leakage_resolved,rationale
0,A_baseline14,14,conditional_parity_audit,False,False,False,False,Exact executable 14D order from notebook 02. I...
1,B_baseline_plus_so_morphology,16,conditional_parity_audit,False,False,False,False,"Adds observable UP/DOWN-proxy durations, not d..."
2,C_baseline_plus_spindle_occupancy,15,diagnostic_reference,False,False,False,False,Occupancy is record-level finite but approxima...
3,D_baseline_plus_event_phase,17,blocked_low_support,False,False,False,False,Each new circular field has only 10/142 valid ...
4,E_recommended_frozen_augmented,not frozen,no_schema_frozen,False,False,False,False,No augmented schema is frozen before a simulat...
5,F_complete23_diagnostic,23,diagnostic_reference,False,False,False,False,The complete candidate library intentionally r...


Freeze artifact directory: D:\Year3_Mao_Projects\sleep_loop\S4_sbi\results\overnight_observation_ablation\schema_freeze


## Go/No-Go

本轮可以冻结候选 feature order 和 rejection rationale，但没有任何 augmented schema 被提升为 SNPE-ready。下一步 04 将验证真实 EEG 与模拟 signal 是否有同名、同单位、同语义的 fixed-length observable contract。

In [4]:
manifest = freeze_result["manifest"]
validation = json.loads((freeze_result["output_dir"] / "validation_report.json").read_text(encoding="utf-8"))
print(json.dumps(manifest["go_no_go"], indent=2))
print(json.dumps(validation, indent=2))
assert validation["artifact_reload"]
assert not validation["pilot_go"]

{
  "pilot_snpe": "NO-GO_PENDING_PARITY",
  "reason": "No schema has a validated simulator-visible EEG extractor, frozen scaling, frozen aggregation, frozen validity handling, and prospective held-out allocation."
}
{
  "artifact_reload": true,
  "augmented_dimension_pass": true,
  "baseline_dimension_pass": true,
  "epoch_accounting_pass": true,
  "pilot_go": false,
  "prefix_identity_pass": true,
  "record_vectors_finite": true,
  "schema_version": "overnight-observation-ablation-v0.1"
}
